In [1]:
import random
import base64
import glob
import os
import json
import re
import uuid

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from object_aware_image_divider import ObjectAwareImageDivider
from openai import OpenAI
from text_to_num import alpha2digit
from tqdm import tqdm

from eval import OBJECT_OF_INTEREST_DETERMINER_PROMPT, OBJECT_OF_INTEREST_COUNTER_PROMPT, AREA_DETECTION_PROMPT

load_dotenv()

/home/muhammad/ghofrani-workspace/visual-reasoning/Grounded-SAM/GroundingDINO/groundingdino/models/GroundingDINO/ms_deform_attn.py:31: UserWarning: Failed to load custom C++ ops. Running on CPU mode Only!
  warnings.warn("Failed to load custom C++ ops. Running on CPU mode Only!")


True

In [2]:
experiment_id = "27f01598"

In [3]:
OPENAI_MODEL_NAME = "gpt-4o"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_MAX_TOKENS = 300
OPENAI_TEMPERATURE = 0.0

EXPERIMENT_ID = experiment_id if experiment_id else uuid.uuid4().hex[:8]
WORKING_DIRECTORY = f"batch_{EXPERIMENT_ID}"

IMAGE_BASE_PATH = "data/cocoqa/images"
DATASET_PATH = "data/cocoqa/cocoqa_count_5more.json"

RANDOM_SEED = 42

In [4]:
df = pd.read_json(DATASET_PATH, orient="records")
df = df.iloc[:40]

In [5]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [6]:
if os.path.exists(WORKING_DIRECTORY):
    with open(f"{WORKING_DIRECTORY}/status.json") as f:
        status = json.load(f)
else:
    status = {
        "object_of_interest": "",
        "image_processing": "",
        "object_counter": "",
    }
    os.mkdir(WORKING_DIRECTORY)
    with open(f"{WORKING_DIRECTORY}/status.json", "w") as f:
        json.dump(
            status,
            f,
            ensure_ascii=False,
            indent=4,
        )
    df.to_json(f"{WORKING_DIRECTORY}/data.json", orient="records")

In [7]:
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)

# Check Batch Status

# Object of Interest determiner

## Creating batches

In [8]:
if not status["object_of_interest"]:
    with open(f"{WORKING_DIRECTORY}/object_of_interest_batch.jsonl", "w") as f:
        for _, row in tqdm(df.iterrows(), total=len(df)):
            prompt = OBJECT_OF_INTEREST_DETERMINER_PROMPT.format(question=row["question"])
            f.write(
                json.dumps(
                    {
                        "custom_id": f"{row['index']}",
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {
                            "model": OPENAI_MODEL_NAME,
                            "messages": [
                                {
                                    "role": "user",
                                    "content": [{"type": "text", "text": prompt}],
                                },
                            ],
                            "max_tokens": OPENAI_MAX_TOKENS,
                            "temperature": OPENAI_TEMPERATURE,
                        },
                    },
                    ensure_ascii=False,
                )
            )
            f.write("\n")

    batch_input_file = openai_client.files.create(
        file=open(f"{WORKING_DIRECTORY}/object_of_interest_batch.jsonl", "rb"),
        purpose="batch",
    )

    batch = openai_client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
            "description": f"object of interest determiner of the expriment {EXPERIMENT_ID}"
        },
    )

    status["object_of_interest"] = batch.to_dict()
    with open(f"{WORKING_DIRECTORY}/status.json", "w") as f:
        json.dump(status, f, ensure_ascii=False, indent=4)

100%|██████████| 50/50 [00:00<00:00, 6281.53it/s]


In [12]:
with open(f"{WORKING_DIRECTORY}/status.json", 'w') as f:
    status["object_of_interest"] = openai_client.batches.retrieve(status["object_of_interest"]["id"]).to_dict()
    json.dump(status, f, indent=4, ensure_ascii=False)

## Parsing Results

In [13]:
if status["object_of_interest"]["status"] == "completed":
    file_response = openai_client.files.content(status["object_of_interest"]["output_file_id"])
    with open(f"{WORKING_DIRECTORY}/object_of_interest_output.jsonl", 'w') as f:
        for line in file_response.iter_lines():
            jsonl = json.loads(line)
            objects_of_interest = ".".join(
                [
                    obj.group(1)
                    for obj in re.finditer(
                        r"\*\*(.+?)\*\*",
                        jsonl["response"]["body"]["choices"][0]["message"]["content"],
                        re.DOTALL,
                    )
                ]
            )
            f.write(json.dumps({
                "custom_id": jsonl["custom_id"],
                "output": objects_of_interest
            }))
            f.write("\n")

# Area Detection & SubImages

In [ ]:
with open(f"{WORKING_DIRECTORY}/object_of_interest_output.jsonl") as f:
    for (_, row), line in zip(df.iterrows(), f):
        jsonl = json.loads(line)
        output_folder = f"{WORKING_DIRECTORY}/{jsonl['custom_id']}"

        object_aware_image_divider = ObjectAwareImageDivider()
        object_aware_image_divider.detect_area(
            image_path=f"{IMAGE_BASE_PATH}/{row['image_url'].split('/')[-1]}",
            prompt=AREA_DETECTION_PROMPT.format(object_of_interest=jsonl["output"]),
            output_folder=output_folder,
        )

        for area_image_path in sorted(glob.glob(f"{output_folder}/**_crops/*.png")):
            output_dir = os.path.splitext(area_image_path)[0]
            os.makedirs(output_dir)
            object_aware_image_divider.divide_by_prompt(
                image_path=area_image_path,
                prompt=jsonl["output"],
                output_folder=output_dir,
                vertical_divides=-1,
                horizontal_divides=0,
            )

# Object Counter

## Creating Batches

In [19]:
with open(f"{WORKING_DIRECTORY}/object_of_interest_output.jsonl") as fi, open(
    f"{WORKING_DIRECTORY}/object_counter_batch.jsonl", "w"
) as fc:
    for line in fi:
        jsonl = json.loads(line)
        if int(jsonl['custom_id']) > 40:
            continue
        for image_path in sorted(
            glob.glob(
                f"{WORKING_DIRECTORY}/{jsonl['custom_id']}/**/**_subimages/*.png",
                recursive=True,
            )
        ):
            with open(image_path, "rb") as f:
                image = f.read()
            fc.write(
                json.dumps(
                    {
                        "custom_id": f"{jsonl['custom_id']}_{image_path.split('/')[-1]}",
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {
                            "model": OPENAI_MODEL_NAME,
                            "messages": [
                                {
                                    "role": "user",
                                    "content": [
                                        {
                                            "type": "text",
                                            "text": OBJECT_OF_INTEREST_COUNTER_PROMPT.format(
                                                object_of_interest=jsonl["output"]
                                            ),
                                        },
                                        {
                                            "type": "image_url",
                                            "image_url": {
                                                "url": f"data:image/png;base64,{base64.b64encode(image).decode('utf-8')}"
                                            },
                                        },
                                    ],
                                },
                            ],
                            "max_tokens": OPENAI_MAX_TOKENS,
                            "temperature": OPENAI_TEMPERATURE,
                        },
                    },
                    ensure_ascii=False,
                )
            )
            fc.write("\n")

    batch_input_file = openai_client.files.create(
        file=open(f"{WORKING_DIRECTORY}/object_counter_batch.jsonl", "rb"),
        purpose="batch",
    )

    batch = openai_client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": f"object counter of the expriment {EXPERIMENT_ID}"},
    )

    status["object_counter"] = batch.to_dict()
    with open(f"{WORKING_DIRECTORY}/status.json", "w") as f:
        json.dump(status, f, ensure_ascii=False, indent=4)

In [31]:
with open(f"{WORKING_DIRECTORY}/status.json", 'w') as f:
    status["object_counter"] = openai_client.batches.retrieve(status["object_counter"]["id"]).to_dict()
    json.dump(status, f, indent=4, ensure_ascii=False)

## Parsing Results

In [22]:
if status["object_counter"]["status"] == "completed":
    file_response = openai_client.files.content(
        status["object_counter"]["output_file_id"]
    )
    with open(f"{WORKING_DIRECTORY}/object_counter_output.jsonl", "w") as f:
        index2count = {}
        for line in file_response.iter_lines():
            jsonl = json.loads(line)
            index = jsonl["custom_id"].split("_", maxsplit=1)[0]

            llm_count = None
            for match in re.finditer(
                r"\*\*(.+?)\*\*",
                jsonl["response"]["body"]["choices"][0]["message"]["content"],
                re.DOTALL,
            ):
                llm_count = alpha2digit(text=match.group(1), lang="en")
                if llm_count.isdigit():
                    if index not in index2count:
                        index2count[index] = 0
                    index2count[index] += int(llm_count)
                    break
            if not llm_count:
                print(jsonl["custom_id"])

        json.dump(index2count, f, indent=4)

# Evaluation

In [31]:
exact_accuracy = 0
mean_absolute_error = 0

for _, row in df.iterrows():
    llm_count = index2count[str(row['index'])]
    gold_count = row['answer']
    exact_accuracy += ((llm_count == gold_count)/len(df))
    mean_absolute_error += (abs(llm_count - gold_count)/len(df))

print(exact_accuracy)
print(mean_absolute_error)

0.37500000000000006
2.2999999999999994
